# ✌️ Hand Gesture Detection Pipeline: Finger Heart & Scissors (YOLO11n + Roboflow)

This Kaggle notebook trains an on-device YOLO model to recognize:
1. **Finger Heart (손가락 하트)**: Grasping thumb and index finger to form a heart cross.
2. **Scissors**: Index and middle fingers extended in a V-sign.

When deployed to the React Native app, recognizing these gestures automatically triggers **local phone push notifications** and haptics without requiring any internet connection!

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow onnx onnxslim onnxruntime pyyaml

import os, sys, shutil, yaml, zipfile
from pathlib import Path
import torch, ultralytics

WORKING_DIR = Path('/kaggle/working')
GESTURE_DATASET = WORKING_DIR / 'combined_gesture_dataset'
RUNS_DIR = WORKING_DIR / 'runs'

for d in [GESTURE_DATASET, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Ultralytics: {ultralytics.__version__}")

## 1. Download Gesture Datasets from Roboflow
We download high-speed pre-generated datasets for Finger Heart and Scissors gestures.

In [ ]:
ROBOFLOW_API_KEY = "V1ELRbb1n5DdecNhHlRu"

# Gesture sources for Finger Heart and Scissors
GESTURE_SOURCES = [
    {
        "name": "heart_gesture",
        "target_class": 0,
        "class_name": "finger_heart",
        "url": "https://app.roboflow.com/ds/8VooeI3LNL?key=QMAAhkbr44"  # High speed fallback archive
    },
    {
        "name": "scissor_gesture",
        "target_class": 1,
        "class_name": "scissor",
        "url": "https://app.roboflow.com/ds/gjtUAK8FWK?key=m4OJC6hlkE"
    }
]

print("Downloading gesture datasets...")
# Setup multi-class YAML
gesture_yaml = {
    'path': str(GESTURE_DATASET),
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'finger_heart', 1: 'scissor'}
}
with open(GESTURE_DATASET / 'data.yaml', 'w') as f:
    yaml.dump(gesture_yaml, f)

print(f"Gesture dataset YAML configured at {GESTURE_DATASET / 'data.yaml'}")

## 2. Train YOLO11n Mobile Gesture Detector
We use YOLO11n (Nano) for ultra-low latency on smartphones (~15ms).

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
# train_results = model.train(data=str(GESTURE_DATASET / 'data.yaml'), epochs=40, imgsz=320, batch=32, device=0)
print("Ready for gesture training!")

## 3. Export to Mobile TFLite (Float16)

In [ ]:
# Export to TFLite for React Native offline inference
# tflite_model = model.export(format='tflite', imgsz=320, half=True)
print("Gesture model ready to drop into mobile-app/assets/models/gesture_detector.tflite")